Setup:

In [1]:
import gc
import torch

torch.cuda.empty_cache()
gc.collect()

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B', dtype=torch.bfloat16, device_map="auto")
model.seqlen = model.config.max_position_embeddings

/home/mrajnoha/double-block-sparse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:11<00:00, 25.53it/s] 


Tests:

In [ ]:
from llm import *

def main():
    model.eval()
    print("Retrieving C4...")
    dataloader, testloader = get_c4(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)
    print("C4 retrieved.")

    tick = time.time()
    print("Running calibration...")
    llama_sequential(model)
    for n, p in model.named_parameters():
        print(n, torch.mean((p == 0).float()))
        if 'down_proj' in n:
            break
    print(time.time() - tick)
    print("Calibration finished.")

    print("Running evaluation...")
    dataloader, testloader = get_wikitext2(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)
    llama_eval(model, testloader)
    print("Evaluation finished, saving model...")

    filepath = "./../pruned/"
    model.save_pretrained(filepath)

# TODO test pipeline
# TODO fix pipeline

In [4]:
main()

Retrieving C4...
C4 retrieved.
Running calibration...
Starting...


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.00 GiB. GPU 0 has a total capacity of 15.74 GiB of which 915.62 MiB is free. Including non-PyTorch memory, this process has 14.80 GiB memory in use. Of the allocated memory 13.28 GiB is allocated by PyTorch, and 1.39 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Cleanup:

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()